# KcELECTRA 파인튜닝 v3_1 — 2026-05-09 -> 이거는 잘못된 것

**담당:** 경이 (kyeongyi)  
**목적:** notice_sample_v6_2_20260509.csv (15948행) 기준으로 재학습.  
핵심: **셀 7 클래스 가중치**를 실제 train 분포에서 동적으로 계산하여  
Simple (TF-IDF + LogReg) 대비 **모든 지표에서** 우위를 확보한다.

---

## v3 vs v3_1 변경점

| 항목 | v3 (20260505) | v3_1 (20260509) |
|---|---|---|
| 데이터 | split_v5_20260505.csv (4992행) | **split_v3_1_20260509.csv (15948행)** |
| train 크기 | 3993개 | **12759개 (3.2배)** |
| 모델 | koelectra-base-v3 | koelectra-base-v3 (동일) |
| LR | 2e-5 | 2e-5 (동일 — 가장 안정적) |
| 손실 함수 | Weighted CE | **Weighted CE (train 분포에서 동적 계산)** |
| 기타 class 비율 | 12.7% | **46.5%** (대폭 증가) |
| 준비물 class 비율 | 4.4% | 2.8% |

## 실행 전 체크리스트

- [ ] 런타임 → **GPU** 선택 (T4 이상)
- [ ] `split_v3_1_20260509.csv` 업로드 (`/content/` 폴더)
- [ ] **런타임 초기화** (런타임 → 세션 다시 시작)
- [ ] 모든 셀 순서대로 실행 (~40분)

## 완료 후 해야 할 일

1. `/content/kcelectra-category-v3_1.zip` 다운로드  
   → 압축 해제 후 로컬 `checkpoints/kcelectra-category-v3_1/`에 배치
2. `/content/eval_results_kcelectra_v3_1_20260509.json` 다운로드  
   → 로컬 `data/20260509/`에 복사
3. `python scripts/evaluate_compare_v3_1_20260509.py` 재실행

In [ ]:
import pandas as pd
df = pd.read_csv("/content/split_v3_1_20260509.csv", encoding="utf-8-sig")
print(f"행 수: {len(df)}")
print(f"컬럼: {df.columns.tolist()}")
print(df["split"].value_counts())
print("\n카테고리 분포 (전체):")
print(df["category"].value_counts())

In [ ]:
# 셀 1: 라이브러리 설치 (Colab에서만 실행)
!pip install transformers datasets seaborn matplotlib scikit-learn -q

In [ ]:
# 셀 2: 시드 고정 + GPU 확인
import os, random
import numpy as np
import torch

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("[경고] GPU 없음 -- CPU로 실행하면 수 시간 걸릴 수 있음")

In [ ]:
# 셀 3: 하이퍼파라미터 설정
#
# v3_1 핵심:
#   - 데이터: 15948행 (v3: 4992행, 3.2배 증가)
#   - 모델: koelectra-BASE v3 (v3와 동일, 110M params)
#   - LR=2e-5 유지 (v4~v6 시도에서 2e-5가 가장 안정적임이 증명됨)
#   - 클래스 가중치: train 분포에서 동적 계산 (셀 7)

from pathlib import Path

_IN_COLAB = os.path.exists("/content")

if _IN_COLAB:
    DATA_DIR  = Path("/content")
    CKPT_DIR  = Path("/content/kcelectra-category-v3_1")
    EVAL_JSON = Path("/content/eval_results_kcelectra_v3_1_20260509.json")
else:
    _BASE     = Path(".").resolve().parent
    DATA_DIR  = _BASE / "data"
    CKPT_DIR  = _BASE / "checkpoints" / "kcelectra-category-v3_1"
    EVAL_JSON = _BASE / "data" / "20260509" / "eval_results_kcelectra_v3_1_20260509.json"

SPLIT_CSV = DATA_DIR / "split_v3_1_20260509.csv"

BASE_MODEL   = "monologg/koelectra-base-v3-discriminator"
MAX_LEN      = 128
BATCH_SIZE   = 16
EPOCHS       = 20
LR           = 2e-5
WARMUP_RATIO = 0.1
PATIENCE     = 5

LABELS   = ["일정", "준비물", "제출", "비용", "건강·안전", "기타"]
LABEL2ID = {l: i for i, l in enumerate(LABELS)}
ID2LABEL = {i: l for i, l in enumerate(LABELS)}

print(f"데이터:         {SPLIT_CSV}")
print(f"체크포인트:     {CKPT_DIR}")
print(f"베이스 모델:    {BASE_MODEL}")
print(f"설정:           {EPOCHS}에폭, LR={LR}, BS={BATCH_SIZE}, MaxLen={MAX_LEN}, patience={PATIENCE}")

In [ ]:
# 셀 4: 데이터 로드
import pandas as pd
from collections import Counter

df = pd.read_csv(SPLIT_CSV, encoding="utf-8-sig")
df = df[df["category"].isin(LABELS)].dropna(subset=["text", "category"])

train_df = df[df["split"] == "train"].reset_index(drop=True)
val_df   = df[df["split"] == "val"].reset_index(drop=True)
test_df  = df[df["split"] == "test"].reset_index(drop=True)

print(f"Train: {len(train_df)}개  (v3: 3993개 -> v3_1: {len(train_df)}개, {len(train_df)/3993:.1f}배 증가)")
print(f"Val:   {len(val_df)}개")
print(f"Test:  {len(test_df)}개")

print("\nTrain 카테고리 분포:")
for lbl, cnt in Counter(train_df["category"]).most_common():
    pct = cnt / len(train_df) * 100
    print(f"  {lbl:8s}: {cnt:5d}개  ({pct:.1f}%)")

In [ ]:
# 셀 5: Dataset + DataLoader 생성
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)


class NoticeDataset(Dataset):
    def __init__(self, df: pd.DataFrame):
        self.texts  = df["text"].tolist()
        self.labels = [LABEL2ID[c] for c in df["category"]]

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = tokenizer(
            self.texts[idx],
            max_length=MAX_LEN,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )
        return {
            "input_ids":      enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "label":          torch.tensor(self.labels[idx], dtype=torch.long),
        }


train_loader = DataLoader(NoticeDataset(train_df), batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(NoticeDataset(val_df),   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(NoticeDataset(test_df),  batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f"DataLoader 생성 완료")
print(f"  train batches: {len(train_loader)}")
print(f"  val   batches: {len(val_loader)}")
print(f"  test  batches: {len(test_loader)}")

In [ ]:
# 셀 6: [핵심] 가중치 손실 함수 계산
#
# v3_1 데이터 불균형 실태:
#   기타 46.5% (5928개) vs 준비물 2.8% (358개) -- 약 16.6배 차이
#
# 공식: weight[i] = total_train / (num_classes * count[i])
# => 소수 클래스(준비물, 비용)에 높은 가중치를 부여해
#    KcELECTRA가 Simple보다 모든 카테고리에서 우수하게 만든다.

label_counts = [int((train_df["category"] == lbl).sum()) for lbl in LABELS]
total = sum(label_counts)
class_weights = torch.tensor(
    [total / (len(LABELS) * c) for c in label_counts],
    dtype=torch.float
).to(DEVICE)

print(f"클래스 가중치 (v3_1, train {total}개):")
for lbl, cnt, w in zip(LABELS, label_counts, class_weights):
    pct = cnt / total * 100
    print(f"  {lbl:8s}: {cnt:5d}개  ({pct:5.1f}%)  가중치={w:.3f}")

print(f"\n불균형 비율 (최대/최소): {max(label_counts)/min(label_counts):.1f}배")
print(f"가중치 최대/최소: {float(class_weights.max()):.3f} / {float(class_weights.min()):.3f}")

loss_fn = torch.nn.CrossEntropyLoss(weight=class_weights)

In [ ]:
# 셀 7: 모델 초기화
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL,
    num_labels=len(LABELS),
    id2label=ID2LABEL,
    label2id=LABEL2ID,
)
model = model.to(DEVICE)

total_params = sum(p.numel() for p in model.parameters())
trainable    = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"모델 파라미터: 전체={total_params:,} / 학습가능={trainable:,}")
print(f"베이스 모델: {BASE_MODEL}")

In [ ]:
# 셀 8: Optimizer + Scheduler
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup

optimizer = AdamW(model.parameters(), lr=LR)

total_steps  = len(train_loader) * EPOCHS
warmup_steps = int(total_steps * WARMUP_RATIO)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps,
)

print(f"전체 스텝: {total_steps}, 워밍업 스텝: {warmup_steps} ({WARMUP_RATIO*100:.0f}%)")
print(f"에폭당 배치 수: {len(train_loader)}")

In [ ]:
# 셀 9: 학습 루프 (조기종료 포함)
from sklearn.metrics import f1_score

def evaluate_epoch(loader):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(DEVICE)
            attn_mask = batch["attention_mask"].to(DEVICE)
            labels    = batch["label"].to(DEVICE)
            logits    = model(input_ids=input_ids, attention_mask=attn_mask).logits
            preds     = logits.argmax(dim=-1)
            all_preds.extend(preds.cpu().tolist())
            all_labels.extend(labels.cpu().tolist())
    return f1_score(all_labels, all_preds, average="macro", zero_division=0)


train_losses, val_f1s = [], []
best_val_f1    = 0.0
patience_count = 0

CKPT_DIR.mkdir(parents=True, exist_ok=True)

for epoch in range(1, EPOCHS + 1):
    model.train()
    epoch_loss = 0.0

    for batch in train_loader:
        input_ids = batch["input_ids"].to(DEVICE)
        attn_mask = batch["attention_mask"].to(DEVICE)
        labels    = batch["label"].to(DEVICE)

        outputs = model(input_ids=input_ids, attention_mask=attn_mask)
        loss    = loss_fn(outputs.logits, labels)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        epoch_loss += loss.item()

    avg_loss = epoch_loss / len(train_loader)
    val_f1   = evaluate_epoch(val_loader)
    train_losses.append(avg_loss)
    val_f1s.append(val_f1)

    print(f"Epoch {epoch:2d}/{EPOCHS} | Loss: {avg_loss:.4f} | Val F1: {val_f1:.4f}", end="")

    if val_f1 > best_val_f1:
        best_val_f1    = val_f1
        patience_count = 0
        model.save_pretrained(CKPT_DIR)
        tokenizer.save_pretrained(CKPT_DIR)
        print(f"  [저장] Best 갱신! ({best_val_f1:.4f})")
    else:
        patience_count += 1
        print(f"  [patience {patience_count}/{PATIENCE}]")
        if patience_count >= PATIENCE:
            print(f"\n조기종료 -- {PATIENCE} 에폭 동안 개선 없음.")
            break

SIMPLE_V3_F1 = 0.8116
print(f"\n학습 완료. Best Val F1: {best_val_f1:.4f}")
print(f"Simple 베이스라인({SIMPLE_V3_F1}) 대비: {best_val_f1 - SIMPLE_V3_F1:+.4f}")

In [ ]:
# 셀 10: label2id.json 저장
import json

label2id_path = CKPT_DIR / "label2id.json"
with open(label2id_path, "w", encoding="utf-8") as f:
    json.dump(LABEL2ID, f, ensure_ascii=False, indent=2)
print(f"label2id.json 저장: {label2id_path}")
print(LABEL2ID)

In [ ]:
# 한글 폰트 설치 및 등록 (셀 11 실행 전에 먼저 실행)
import subprocess
import matplotlib
import matplotlib.font_manager as fm

subprocess.run(["apt-get", "install", "-y", "fonts-nanum"], capture_output=True)

font_path = "/usr/share/fonts/truetype/nanum/NanumGothic.ttf"
fm.fontManager.addfont(font_path)
prop = fm.FontProperties(fname=font_path)
matplotlib.rc("font", family=prop.get_name())
matplotlib.rcParams["axes.unicode_minus"] = False

print(f"등록된 폰트: {prop.get_name()}")
print("한글 테스트: 일정 준비물 제출 비용 건강·안전 기타")

In [ ]:
# 셀 11: 학습 곡선 시각화
import matplotlib.pyplot as plt
import platform

if platform.system() == "Windows":
    matplotlib.rc("font", family="Malgun Gothic")
elif platform.system() == "Darwin":
    matplotlib.rc("font", family="AppleGothic")

matplotlib.rcParams["axes.unicode_minus"] = False

actual_epochs = len(train_losses)
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].plot(range(1, actual_epochs+1), train_losses, marker="o", color="steelblue", linewidth=2)
axes[0].set_title("학습 손실 (Train Loss) -- KcELECTRA v3_1", fontsize=13)
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].grid(True, alpha=0.3)

axes[1].plot(range(1, actual_epochs+1), val_f1s, marker="o", color="darkorange", linewidth=2)
axes[1].axhline(y=best_val_f1, color="red", linestyle="--", alpha=0.7, label=f"Best={best_val_f1:.4f}")
axes[1].axhline(y=0.8116, color="gray", linestyle=":", alpha=0.6, label="Simple 베이스라인=0.8116")
axes[1].set_title("검증 Macro F1 -- KcELECTRA v3_1", fontsize=13)
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Macro F1")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
curve_path = DATA_DIR / "train_curve_kcelectra_v3_1_20260509.png"
plt.savefig(curve_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"학습 곡선 저장: {curve_path}")

In [ ]:
# 셀 12: Test 세트 최종 평가
from sklearn.metrics import classification_report, confusion_matrix, f1_score

best_model = AutoModelForSequenceClassification.from_pretrained(
    CKPT_DIR, num_labels=len(LABELS)
).to(DEVICE)
best_model.eval()

all_preds, all_labels = [], []
with torch.no_grad():
    for batch in test_loader:
        input_ids = batch["input_ids"].to(DEVICE)
        attn_mask = batch["attention_mask"].to(DEVICE)
        labels    = batch["label"].to(DEVICE)
        logits    = best_model(input_ids=input_ids, attention_mask=attn_mask).logits
        preds     = logits.argmax(dim=-1)
        all_preds.extend(preds.cpu().tolist())
        all_labels.extend(labels.cpu().tolist())

pred_names = [ID2LABEL[p] for p in all_preds]
true_names = [ID2LABEL[l] for l in all_labels]
macro_f1   = f1_score(all_labels, all_preds, average="macro", zero_division=0)

SIMPLE_V3_F1 = 0.8116

print("[KcELECTRA v3_1] Test 세트 최종 평가")
print("=" * 55)
print(classification_report(true_names, pred_names, labels=LABELS, zero_division=0))
print(f"Macro F1             : {macro_f1:.4f}")
print(f"Simple v3 대비       : {macro_f1 - SIMPLE_V3_F1:+.4f}  (기준: {SIMPLE_V3_F1})")
if macro_f1 > SIMPLE_V3_F1 + 0.05:
    print(f"\n=> KcELECTRA v3_1이 Simple 대비 5%+ 향상 -- 채택!")
elif macro_f1 > SIMPLE_V3_F1:
    print(f"\n=> KcELECTRA v3_1이 Simple보다 향상됨")
else:
    print(f"\n=> 아직 Simple보다 낮음")

In [ ]:
# 셀 13: Confusion Matrix
import seaborn as sns

cm = confusion_matrix(true_names, pred_names, labels=LABELS)

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=LABELS, yticklabels=LABELS, ax=ax)
ax.set_xlabel("예측 카테고리")
ax.set_ylabel("실제 카테고리")
ax.set_title(f"KcELECTRA v3_1 Confusion Matrix (Macro F1={macro_f1:.4f})")
plt.tight_layout()
cm_path = DATA_DIR / "confusion_matrix_kcelectra_v3_1_20260509.png"
plt.savefig(cm_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Confusion Matrix 저장: {cm_path}")

In [ ]:
# 셀 14: 평가 결과 JSON 저장
# evaluate_compare_v3_1_20260509.py가 이 JSON을 읽어 비교 분석.
# 로컬 model/classification/data/20260509/ 에 복사해야 할 파일!

report = classification_report(
    true_names, pred_names, labels=LABELS, output_dict=True, zero_division=0
)

result = {
    "model":            "kcelectra",
    "version":          "v3_1",
    "macro_f1":         round(macro_f1, 4),
    "macro_precision":  round(report["macro avg"]["precision"], 4),
    "macro_recall":     round(report["macro avg"]["recall"], 4),
    "per_class": {
        lbl: {
            "precision": round(report[lbl]["precision"], 4),
            "recall":    round(report[lbl]["recall"], 4),
            "f1":        round(report[lbl]["f1-score"], 4),
            "support":   report[lbl]["support"],
        }
        for lbl in LABELS
    },
    "confusion_matrix": cm.tolist(),
    "labels":           LABELS,
    "data_version":     "v3_1_20260509",
    "train_size":       len(train_df),
    "test_size":        len(test_df),
    "epochs_trained":   len(train_losses),
    "best_val_f1":      round(best_val_f1, 4),
    "lr":               LR,
    "batch_size":       BATCH_SIZE,
    "warmup_ratio":     WARMUP_RATIO,
    "weighted_loss":    True,
    "class_weights":    [round(float(w), 4) for w in class_weights.cpu()],
    "early_stopping":   True,
    "patience":         PATIENCE,
}

EVAL_JSON.parent.mkdir(parents=True, exist_ok=True)
with open(EVAL_JSON, "w", encoding="utf-8") as f:
    json.dump(result, f, ensure_ascii=False, indent=2)

print(f"[저장] {EVAL_JSON}")
print(f"\n★ 이 파일을 로컬 model/classification/data/20260509/ 에 복사하세요!")
print(f"★ kcelectra-category-v3_1/ 폴더도 checkpoints/ 에 복사하세요!")

In [ ]:
# 셀 15: 다운로드용 압축
import shutil

shutil.make_archive("/content/kcelectra-category-v3_1", "zip", "/content/kcelectra-category-v3_1")
print("체크포인트 압축 완료: /content/kcelectra-category-v3_1.zip")
print("\n다운로드 목록:")
print("  1. /content/kcelectra-category-v3_1.zip")
print("  2. /content/eval_results_kcelectra_v3_1_20260509.json")
print("  3. /content/train_curve_kcelectra_v3_1_20260509.png")
print("  4. /content/confusion_matrix_kcelectra_v3_1_20260509.png")

## 학습 완료 후 로컬 작업

```bash
# 1. 파일 배치
#    kcelectra-category-v3_1/ -> model/classification/checkpoints/
#    eval_results_kcelectra_v3_1_20260509.json -> model/classification/data/20260509/
#    *.png -> model/classification/data/20260509/

# 2. 비교 평가 실행
cd model/classification
python scripts/evaluate_compare_v3_1_20260509.py

# 3. 시각화 노트북 실행
jupyter notebook notebooks/12_visualize_comparison_v3_1_20260509.ipynb
```